## 什么叫“依赖”
一个接口需要知道当前登录用户,这个“当前用户”就是路由函数的依赖。
## 什么是依赖注入？
函数声明自己需要什么，但不负责亲自创建它；由外部系统创建后注入进来。

In [ ]:
from typing import Annotated
from fastapi import Depends


@app.get("/profile")
async def get_profile(
    current_user: Annotated[
        User,
        Depends(get_current_user),
    ],
):
    return current_user

### Depends 中传的是函数对象
表示交给 FastAPI 在请求到来时调用。

Depends() 接收的是一个可调用对象，不是提前执行后的结果。

可调用对象可以是：
- 普通函数
- 异步函数
- 类
- 实现了 __call__ 的对象

### 为什么推荐使用 Annotated
Annotated 可以理解为：
- 这个参数的 Python 类型是 User
- 同时附加一条 FastAPI 元数据：
- 它的值来自 Depends(get_current_user)
```python
from typing import Annotated
from fastapi import Depends
CurrentUser = Annotated[
    User,
    Depends(get_current_user),
]
@app.get("/profile")
async def get_profile(
    current_user: CurrentUser,
):
    return current_user
```

### FastAPI 如何解析依赖？

In [ ]:
async def get_token(
    authorization: str,
) -> str:
    return authorization


async def get_current_user(
    token: Annotated[str, Depends(get_token)],
) -> User:
    return await find_user_by_token(token)


@app.get("/profile")
async def get_profile(
    user: Annotated[
        User,
        Depends(get_current_user),
    ],
):
    return user

FastAPI 会建立依赖树，从最底层开始解析：
1. 执行 get_token
2. 将 token 注入 get_current_user
3. 执行 get_current_user
4. 将 user 注入 get_profile
5. 执行 get_profile

这叫作子依赖。

### 依赖失败会发生什么？

依赖执行失败:
- → 后续依赖停止
- → 路由函数不执行
- → 异常处理器生成响应

### 一次请求内的依赖缓存

In [ ]:
async def require_active_user(
    user: Annotated[
        User,
        Depends(get_current_user),
    ],
):
    if not user.is_active:
        raise HTTPException(status_code=403)

    return user


async def load_preferences(
    user: Annotated[
        User,
        Depends(get_current_user),
    ],
):
    return await find_preferences(user.id)

@app.get("/dashboard")
async def dashboard(
    user: Annotated[
        User,
        Depends(require_active_user),
    ],
    preferences: Annotated[
        dict,
        Depends(load_preferences),
    ],
):
    return {
        "user": user,
        "preferences": preferences,
    }

"""
dashboard
├── require_active_user
│   └── get_current_user
└── load_preferences
    └── get_current_user
"""

虽然 get_current_user 出现在两条路径上，但默认情况下，同一次请求中只执行一次。

FastAPI 会：
1. 第一次执行 get_current_user
2. 缓存返回值
3. 后续依赖复用该值

In [ ]:
### use_cache=False
# 确实需要同一次请求内重新执行

value: Annotated[
    str,
    Depends(get_value, use_cache=False),
]

### 只执行依赖，不接收返回值

有些依赖只负责检查，不需要把返回值传给路由,可以写在装饰器上

In [ ]:
async def verify_api_key(
    x_api_key: str,
):
    if x_api_key != "secret":
        raise HTTPException(status_code=401)

@app.get(
    "/internal",
    dependencies=[Depends(verify_api_key)],
)
async def internal_api():
    return {"status": "ok"}

### 依赖注入和 Middleware 的区别

两者都能在路由前执行，但作用范围不同。

# Middleware
包裹整个 HTTP 请求
- 所有路由
- 所有响应
- 请求前和响应后
适合：
- Request ID
- 全局日志
- CORS
- 接口耗时
- 链路追踪

# 依赖注入
围绕具体路由需求组织：
- 某个路由
- 一组路由
- 具有返回值
- 可以形成依赖树


适合：
- 当前用户
- 权限验证
- 分页参数
- 数据库 Session
- Repository
- 租户上下文

#### 判断方法
如果它是几乎所有 HTTP 请求共有的横切处理，考虑 Middleware；如果它是某些路由运行所需的前置条件或资源，考虑 Dependency。

### 面试表达
> FastAPI 的依赖注入基于函数签名。路由使用 Depends 声明所需的可调用对象，FastAPI 会递归构建并解析依赖树，将返回值注入路由参数。同一个依赖在一次请求中默认只执行一次并复用结果，这有利于认证、数据库 Session 和请求上下文的一致性，也能通过 dependency override 方便地进行测试替换。